# Data types

This page considers the data types in clickhouse.

## DateTime

The dates in clickhouse there are types that allows to represent time in clickhouse:

| Type            | Storage | Range                         | Precision                   |
| --------------- | ------- | ----------------------------- | --------------------------- |
| `Date`          | 2 bytes | 1970-01-01 – 2149-06-06       | Day                         |
| `Date32`        | 4 bytes | 1900-01-01 – 2299-12-31       | Day                         |
| `DateTime`      | 4 bytes | 1970-01-01 – 2106-02-07       | Second                      |
| `DateTime64(p)` | 8 bytes | 1900-01-01 – 2299-12-31       | Up to nanoseconds (`p=0–9`) |
| `Time`          | 4 bytes | 00:00:00 – 23:59:59           | Second                      |
| `Time64(p)`     | 8 bytes | 00:00:00 – 23:59:59.999999999 | Up to nanoseconds (`p=0–9`) |

The most usefull functions/operators to work with dates:

- The `toDate` function converts a string literar in the format `"yyyy-mm-dd"` into a `Date` object.
- The `dateDiff` function allows to count the period between two dates in the selected units. The minus operator `<date1> - <date2>` is not documented, but it appears to works in the same way as the `dateDiff('days', <date2>, <date1>)` function.
- The `INTERVAL <number> <SECOND | MINUTE | HOUR ...>` allows to create a special interval object that could be added to the date.
- The `now()` function return `DateTime` object that corresponds to current datetime.

May find usefull:
- [Operators for Working with Dates and Times](https://clickhouse.com/docs/sql-reference/operators#operators-for-working-with-dates-and-times).

---

Here are the example of usage of the most usable facilities associated with dates in clickhouse:

In [19]:
--ClickHouse
SELECT
    toDate('2025-08-01') AS date1,
    toDate('2025-07-01') AS date2,
    date1 - date2 AS diff,
    dateDiff('month', date2, date1) AS months,
    date1 + INTERVAL 10 DAY add_interval,
    now();

date1,date2,diff,months,add_interval,now()
2025-08-01,2025-07-01,31,31,2025-08-11,2026-06-08 19:06:43


## Array(T)

Array in clickhouse is a set of objects of the same type.

- Define array using `Array(<type>)`.
- Create inline array using `array(<el1>, <el2>, ...)` or using `[el1, el2, ...]`.
- Use the `sizeN` subcolumn to get the length of the array. The `N` determines the dimention of the array you're interested in. 

Check the description in the [Array(T)](https://clickhouse.com/docs/sql-reference/data-types/array) page of the official documentation.

---

The following cell defines the CTE containing the inline arrays and shows the size of their different diemntions.

In [1]:
--ClickHouse
WITH t_arr (arrays) AS(
    SELECT * FROM VALUES (
        [[1, 2, 3], [2, 3]]
    )
)
SELECT
    arrays.size0 AS first_diemntion,
    arrays.size1 AS second_dimention
FROM t_arr;

first_diemntion,second_dimention
2,"[3, 2]"


## Tuple(T1, T2, ...)

A tuple is an ordered set of elements, each of which has a special data type.

- Define a column to have a `Tuple` data type using syntax `Typle(<type1>, <type2>, ...)`. You can assign a names to the elements of the typle using syntax `Typle(<name1> <type1>, <name2> <type2>, ...)`.
- Create inline tuple with `typle(<val1>, <val2>, ...)` or `(<val1>, <val2>, ...)`.
- Access the elements of a tuple using `.` operator, specifying the name or index of the element. For example, use `typ.1` to access the first element of the tuple, or `typ.a` to access the element named `a`.

Check the official description in the [Tuple](https://clickhouse.com/docs/sql-reference/data-types/tuple) page of the official documentation.

---

The following cell defines the typle in the CTE and accesses its elements.

In [21]:
--ClickHouse
WITH tuple_example AS (
    SELECT ('hello', 10) AS the_tuple
)
SELECT the_tuple, the_tuple.1, the_tuple.2 FROM tuple_example;

the_tuple,"tupleElement(the_tuple, 1)","tupleElement(the_tuple, 2)"
"('hello', 10)",hello,10


## Map(K, V)

Clickhouse implements mappings as an array of tuples: `Array(Tuple(T_key, T_val))`, which means their keys are not unique.

- Define column type with `Map(<key_type>, <value type>)`.
- Access elements using `[]` operator.
- Define the map inline using `map(<key1>, <value1>, <key2>, <value2>, ...)` or `{<key1>: <value1>, <key2>: <value2> ...}`.
- The `keys` and `values` columns return the keys and values `Array`.

---

The following cell defines a ClickHouse map and demonstrates some fundamental operations in queries.

In [8]:
--ClickHouse
WITH map_example AS (
    SELECT map('one', 1, 'two', 2) AS my_map
)
SELECT my_map['one'], my_map.keys, my_map.values FROM map_example;

"arrayElement(my_map, 'one')",my_map.keys,my_map.values
1,"['one', 'two']","[1, 2]"
